# Pause-based Metrics (*CRITT server version*):

## [CRITT Academy](https://github.com/Critt-Kent/CRITT-Academy) Module 01: Lesson 05

### What You Will Do In This Lesson

1. Install and import all necessary Python libraries
2. Load Segment and Keystroke Data tables (SG and KD tables) from the TPR-DB for a single study (a public study) and then read them into a DataFrame
3. Add pause-based metrics based on a different pause threshold than is available by default in the SG tables
4. Calculate additional, derived pause-based metrics such as pause-to-word ratio and average pause ratio

### First time working with a CRITT Academy code notebook?

If you haven't yet gone through the [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR_DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb), you should do that first (this will help you understand Steps 1 through 3 much better).

## Step 1. Import all necessary Python libraries

- **NumPy**: `numpy` is used for scientific computing and data science
- **Pandas**: `pandas` is used for data manipulation, organization, and analysis
- **SciPy `stats`**: the `stats` module within `scipy` is used for statistical analysis and probability distributions
- **Matplotlib**: `matplotlib` is used for data visualization
- **Seaborn**: `seaborn` is used for nice-looking statistical graphics
- **tprdb-utilities**: `tprdb_utilities` is used for getting data remotely from the CRITT TPR-DB and then for reading data tables from a certain study (or studies) into a Pandas DataFrame.

In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "0.7.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tprdb_utilities

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "numpy>=2.4.0" "pandas>=3.0.0" "scipy>=1.17.0" "matplotlib>=3.10.0" "seaborn>=0.13.0" "tprdb-utilities>=0.7.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

In [ ]:
# When working on the CRITT server: set the path to the tprdb_utilities directory:

import sys
sys.path.append(f"/data/tprdb-utilities/src")  # point to the folder that CONTAINS the utilities package

In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    # TPR-DB utilities import
    from tprdb_utilities import read_TPRDB_tables, recompute_pause_based_metrics

    print("✅ All imports successful!")

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 2: Get Segments and Keystroke Data Tables and load them into DataFrames

In [ ]:
# read the Segments (SG) and Keystroke Data (KD) tables into a DataFrame

# this is the path to the TPR-DB on the CRITT server
mothership = '/data/critt/tprdb/'

# BML12: (English) multiLing texts translated into Spanish
studies=["BML12"]

# read the Segments (SG)
SG = read_TPRDB_tables(
    path=mothership,
    user='PUBLIC',
    studies=studies,
    extension="sg",
    verbose=1,
)

# read the Keystroke Data (KD)
KD = read_TPRDB_tables(
    path=mothership,
    user='PUBLIC',
    studies=studies,
    extension="kd",
    verbose=1,
)

### Take a look at the pause-based metrics

In this next code cell, you'll take a look at *everything* in the Segments (SG) and Keystroke Data (KD) DataFrames, but take a *close look* at the Columns in the Segments DataFrame that start with these letters:

* **TB** – **T**yping **B**ursts (the number of stretches of typing used to translate/edit each segment)
* **TG** – **T**yping **G**ap (amount of time, in milliseconds, spent pausing between typing bursts)
* **TD** – **T**yping **D**uration (amount of time, in milliseconds, spent typing)

All of the preceding metrics depend on a certain **pause threshold**. We have to define how long someone has to **not be typing** in order for it to be considered a pause. There are three pause thresholds that come built in to the Segments (SG) tables:

* **1000**: 1000 milliseconds (nice and arbitrary 🤓)
* **KUI**: Keystroke Unit Interruption
* **PUB**: Production Unit Break

To read more about **KUI** and **PUB**, see the [TPR-DB Documentation: Features List](https://critt-kent.github.io/TPR-DB-documentation/analyze/features/).

In [ ]:
# Check the two DataFrames' shapes and display the first few rows (with all columns visible) of each DataFrame
print(f"📊 The Segments DataFrame has {SG.shape[0]} rows and {SG.shape[1]} columns")

with pd.option_context("display.max_columns", None):
    display(SG.head())

print(f"📊 The Keystroke Data table has {KD.shape[0]} rows and {KD.shape[1]} columns")

with pd.option_context("display.max_columns", None):
    display(KD.head())

## Step 3: Recompute the pause-based metrics based on a custom pause threshold

We can use a function called `recompute_pause_based_metrics()` (from the `transformer` module of the [`tprdb-utilities` python library](https://pypi.org/project/tprdb-utilities/)) to add new Typing Burst, Typing Gap, and Typing Duration metrics that are based on a custom pause threshold that we define to the Segments (SG) DataFrame.

In [ ]:
# view help documentation first
recompute_pause_based_metrics??

In [ ]:
# add pause-based metrics using a pause threshold of 500 milliseconds
SG = recompute_pause_based_metrics(SG, KD, 5000)

In [ ]:
# view the columns that were added to the Segments DataFrame
with pd.option_context("display.max_columns", None):
    display(SG.head())

## Step 4: Calculate derivative pause-based metrics

Measures according to Isabel Lacruz and Gregory M. Shreve (2015).
To read more about pause-based metrics, see the [TPR-DB Documentation: Keystroke Pauses](https://critt-kent.github.io/TPR-DB-documentation/analyze/process/#keystroke-pauses).

- $\text{Pause Ratio (PR) } = \frac{\text{total pause time in segment}}{\text{total time in segment}}$

- $\text{Average Pause Ratio (APR)} = \frac{\text{average time per pause}}{\text{average time per words}}$

- $\text{Pause-to-word Ratio (PWR)} = \frac{\text{number of pauses in segment}}{\text{number of words in segment}}$

- $\text{Event-to-Word Ratio (EWR)} =\frac{\text{number of complete editing events}}{\text{number of words}}$


### Compute APR values

In [ ]:
# APR: average time per pause / average time per word
#  average time per word: TypingDuration / number of words: SG["TDx"] / SG["TokT"]
#  average time per pause: TypingGap / Typing bursts + 1 : SG2["TGx"] / SG["TBx"] +1
#  -- there is one more typing pause than typing bursts 

SG["APR5000"] = (SG["TG5000"] * SG["TokT"]) / (1+ SG["TB5000"] * SG["TD5000"] )
SG["APR1000"] = (SG["TG1000"] * SG["TokT"]) / (1+ SG["TB1000"] * SG["TD1000"])
SG["APR_PUB"] = (SG["TG_PUB"] * SG["TokT"]) / (1+ SG["TB_PUB"] * SG["TD_PUB"])
SG["APR_KUI"] = (SG["TG_KUI"] * SG["TokT"]) / (1+ SG["TB_KUI"] * SG["TD_KUI"])

# alternative assumptions:
# --- take into account the PreGap, the first pause in the segment

# SG["APR5000_PG"] = (SG["PreGap"] + SG["TG5000"] * SG["TokT"]) / (1+ SG["TB5000"] * SG["TD5000"] )
# SG["APR1000_PG"] = (SG["PreGap"] + SG["TG1000"] * SG["TokT"]) / (1+ SG["TB1000"] * SG["TD1000"])

In [ ]:
# plot the APR values in a histplot

sns.histplot(data = SG[['APR5000', 'APR1000']], bins=80, log_scale=True)
plt.show()
sns.histplot(data = SG[['APR_PUB', 'APR_KUI']], bins=80, log_scale=True)
plt.show()

with pd.option_context("display.max_columns", None):
    display(SG.head())

In [ ]:
# print correlation between the APR values

SG[['APR5000', 'APR1000', 'APR_PUB', 'APR_KUI']].corr()


### Compute PWR values

In [ ]:
# calculate different Pause-to-Word Ratios (PWR) using different pause thresholds and add them to the Segments (SG) DataFrame
# assuming that:
# -- the number of pauses is identical to the number of bursts, i.e., each burst is preceeded by a pause 
# -- we take "number of words" to be the number of ST words

SG["PWR1000"] = SG["TB1000"] / SG["TokS"]
SG["PWR_KUI"] = SG["TB_KUI"] / SG["TokS"]
SG["PWR_PUB"] = SG["TB_PUB"] / SG["TokS"]
SG["PWR5000"] = SG["TB5000"] / SG["TokS"]

# alternative assumptions are:
# -- there is also a pause following the last burst in a segment: add 1 to nominator:
# -- we take "number of words" to be the number of TT words

# SG["PWR1000_T"] = (SG["TB1000"] + 1) / SG["TokT"]
# SG["PWR_KUI_T"] = (SG["TB_KUI"] + 1) / SG["TokT"]
# SG["PWR_PUB_T"] = (SG["TB_PUB"] +1) / SG["TokT"]
# SG["PWR500_T"] = (SG["TB500"] +) / SG["TokST"]

# alternative assumptions are:
# -- average number of words in the ST and the TT side od the 

# SG["PWR1000_AV"] = 2 * SG["TB1000"] / (SG["TokS"] + SG["TokT"]) 
# SG["PWR_KUI_AV"] = 2 * SG["TB_KUI"] / (SG["TokS"] + SG["TokT"])
# SG["PWR_PUB_AV"] = 2 * SG["TB_PUB"] / (SG["TokS"] + SG["TokT"])
# SG["PWR500_AV"] = 2 * SG["TB500"] / (SG["TokS"] + SG["TokST"])

In [ ]:
# 

ax = sns.histplot(data = SG[['PWR5000', 'PWR1000', 'PWR_PUB', 'PWR_KUI']], bins=80, log_scale=True)

# add labels
ax.set(ylabel='Number of Occurrances', xlabel='Task', title='Number of Pauses per Word per Pause-Threshold')
plt.show()

In [ ]:
# print correlation between the PWR values:

SG[['PWR5000', 'PWR1000', 'PWR_PUB', 'PWR_KUI']].corr()
